# Anomaly Detection Industriale con Autoencoder Convoluzionali
**Studente:** Nicolae Balaban

## 1. Definizione del Problema e Dataset
In ambito manifatturiero, identificare difetti sui prodotti è cruciale. Tuttavia, gli approcci di classificazione supervisionata standard falliscono perché si hanno a disposizione migliaia di immagini di prodotti "sani" e pochissimi esempi di difetti.

Questo progetto affronta il problema tramite **Unsupervised Anomaly Detection**. Utilizzeremo un **Autoencoder Convoluzionale** addestrato esclusivamente su immagini perfette per fargli apprendere la distribuzione normale dei dati. In fase di test, le anomalie emergeranno come errori di ricostruzione.

**Dataset:** MVTec AD (Categoria: `bottle`).
*Nota: Il dataset non è incluso nel repository per limiti di dimensione. Può essere scaricato da Kaggle ed estratto nella cartella `data/bottle/`.*

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve
import torchvision.transforms as T
from torch.utils.data import DataLoader
from pathlib import Path

from utils.visual_util import ColoredPrint as cp
from utils.mvtec_dataset import MVTecDataset
from nets.simple_autoencoder import SimpleAutoencoder
from utils.dataset_analyzer import analyze_mvtec_category

BASE_DIR = Path.cwd()
DATA_ROOT = BASE_DIR / "data"
CATEGORY = "bottle"
MODEL_PATH = BASE_DIR / "runs" / "base_autoencoder.pth"

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
cp.cyan(f"Device: {device}")

In [ ]:
dataset_path = BASE_DIR / "data" / CATEGORY
analyze_mvtec_category(dataset_path)

## 2. Definizione del Problema
Come evidenziato dall'EDA, il dataset è **fortemente sbilanciato**. Abbiamo centinaia di esempi di bottiglie perfette nel training set, ma pochissimi esempi di difetti (e molto vari tra loro) nel test set. 

Un approccio di classificazione supervisionata standard fallirebbe per mancanza di dati sulle classi anomale.
Il problema viene quindi riformulato come **Unsupervised Anomaly Detection**:
1. Verra' addestrato un modello (Autoencoder) **solo** sulle immagini sane.
2. La rete imparerà a comprimere e ricostruire esclusivamente bottiglie perfette.
3. In fase di test, passando un'immagine difettosa, la rete fallirà nel ricostruire il difetto.
4. La differenza (Mean Squared Error) tra l'immagine originale e quella ricostruita genererà una *Anomaly Map*, permettendoci di isolare il difetto.

## 3. Costruzione e Valutazione della Baseline
Qui viene testato un `SimpleAutoencoder` costituito da semplici layer di convoluzione (Conv2d) e deconvoluzione (ConvTranspose2d).
Verrano valutate le performance calcolando il **Mean Squared Error (MSE)** pixel per pixel e calcolando la metrica **AUROC** (Area Under the Receiver Operating Characteristic curve) estraendo il pixel con l'errore massimo per ogni immagine.

In [ ]:
# 1. Caricamento del modello Baseline
model = SimpleAutoencoder().to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device, weights_only=True))
model.eval() # Modalità inferenza

# 2. Setup del DataLoader di Test (contiene sia bottiglie sane che rotte)
transform_pipeline = T.Compose([
    T.Resize((256, 256)),
    T.ToTensor()
])

test_dataset = MVTecDataset(root_dir=DATA_ROOT, category=CATEGORY, is_train=False, transform=transform_pipeline)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

y_true = [] # Le etichette reali (0 = buone, 1 = anomalia)
y_scores = [] # Il punteggio di anomalia predetto

cp.yellow(f"Evaluating of {len(test_dataset)} test images...")

with torch.no_grad():
    for images, labels, paths in test_loader:
        images = images.to(device)

        reconstructed = model(images)

        # Calcoliamo l'errore quadratico (MSE) per ogni pixel nella singola immagine
        mse_map = (images - reconstructed) ** 2

        # Calcoliamo l'Anomaly Score: prendiamo la media degli errori per canale,
        # e poi il valore MASSIMO di tutta la mappa dell'immagine.
        # Se c'è un'anomalia, ci sarà almeno un pixel con un errore altissimo.
        mean_channels = torch.mean(mse_map, dim=1)
        anomaly_score = torch.max(mean_channels).item()

        y_true.append(labels.item())
        y_scores.append(anomaly_score)

auroc = roc_auc_score(y_true, y_scores)

cp.yellow("\n--- EVALUATION RESULT ---")
cp.cyan(f"Dataset: {CATEGORY.upper()}")
cp.cyan(f"AUROC Score: {auroc:.4f} ({auroc * 100:.2f}%)")

y_true_np = np.array(y_true)
y_scores_np = np.array(y_scores)
cp.cyan(f"Average score error to reconstruct GOOD images: {np.mean(y_scores_np[y_true_np == 0]):.4f}")
cp.cyan(f"Average score error to reconstruct ANOMALOUS images: {np.mean(y_scores_np[y_true_np == 1]):.4f}")


fpr, tpr, thresholds = roc_curve(y_true, y_scores)
plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {auroc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlabel('False Positive Rate (Falsi Allarmi)')
plt.ylabel('True Positive Rate (Difetti Trovati)')
plt.title('Curva ROC - Baseline Autoencoder')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()

### 3.1 Dimostrazione della ricostruzione dell'immagine del `SimpleAutoencoder`
I numeri mostrano che il modello riesce a individuare le anomalie, ma per comprendere i limiti del `SimpleAutoencoder` è fondamentale un'ispezione visiva.
Di seguito peschiamo un'immagine anomala dal test set e confrontiamo l'immagine originale, la ricostruzione generata dalla rete e la mappa di calore dell'errore (Anomaly Map).

In [ ]:
from visualize_reconstruction import visualize_reconstruction

visualize_reconstruction()

## 4. Ottimizzazione dell'Algoritmo di ML
Il risultato della baseline (circa 78% AUROC) mostra che il modello riesce a isolare le anomalie, ma fatica a ricostruire i dettagli ad alta frequenza (come i bordi della bottiglia), generando falsi positivi. La forbice tra l'errore medio delle immagini sane e quello delle immagini anomale è troppo stretta.

Qui verra usata una architettura ottimizzata (`OptimizedAutoencoder`) per analizzare se il nuovo autoencoder riesce a gestire meglio i bordi o le anomalie:
1. **Aumento della capacità:** I filtri passano da `[16, 32, 64]` a `[32, 64, 128]` per catturare feature più complesse.
2. **Batch Normalization:** Aggiunta dopo ogni layer convoluzionale per stabilizzare i gradienti e accelerare la convergenza.
3. **LeakyReLU:** Sostituisce la standard ReLU per evitare il problema dei "dying neurons" durante la compressione.


In [ ]:
import torch.optim as optim

from nets.optimized_autoencoder import OptimizedAutoencoder

cp.purple("--- STARTING OPTIMIZATION TRAINING ---")

EPOCHS = 15
BATCH_SIZE = 16
LEARNING_RATE = 1e-3

optimized_model = OptimizedAutoencoder().to(device)
criterion = torch.nn.MSELoss()
optimizer = optim.Adam(optimized_model.parameters(), lr=LEARNING_RATE)

train_dataset = MVTecDataset(root_dir=DATA_ROOT, category=CATEGORY, is_train=True, transform=transform_pipeline)
train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        pin_memory=True,
        num_workers=8,
        prefetch_factor=2)

# Loop di Addestramento
optimized_model.train()
for epoch in range(EPOCHS):
    running_loss = 0.0
    for images, _, _ in train_loader:
        images = images.to(device, non_blocking=True)

        optimizer.zero_grad()
        reconstructed = optimized_model(images)
        loss = criterion(reconstructed, images)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)
    cp.cyan(f"Epoch [{epoch+1}/{EPOCHS}] - Reconstruction Loss: {epoch_loss:.6f}")

print("Training complete! Evaluating optimized model...")

# Valutazione Immediata
optimized_model.eval()
y_true_opt = []
y_scores_opt = []

with torch.no_grad():
    for images, labels, _ in test_loader:
        images = images.to(device)
        reconstructed = optimized_model(images)
        mse_map = (images - reconstructed) ** 2
        anomaly_score = torch.max(torch.mean(mse_map, dim=1)).item()

        y_true_opt.append(labels.item())
        y_scores_opt.append(anomaly_score)

# Calcolo nuova AUROC
auroc_opt = roc_auc_score(y_true_opt, y_scores_opt)

cp.purple("\n--- OPTIMIZED MODEL RESULTS ---")
cp.cyan(f"New AUROC Score: {auroc_opt:.4f} ({auroc_opt * 100:.2f}%)")

# Plot comparativo (Baseline vs Ottimizzata)
fpr_opt, tpr_opt, _ = roc_curve(y_true_opt, y_scores_opt)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='gray', lw=2, linestyle=':', label=f'Baseline (AUROC = {auroc:.3f})')
plt.plot(fpr_opt, tpr_opt, color='green', lw=2, label=f'Optimized (AUROC = {auroc_opt:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison - MVTec AD (Bottle)')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()